# Fixed-source boundary orbit — explicit prerequisite retry
CPU/high RAM. Same Lean source as failed v1; explicit ReflectionOrbitAlgebra prerequisite. Diagnostic only. One execution, preserve first error.

In [ ]:
import hashlib, pathlib, subprocess, datetime, psutil, urllib.request, json, tarfile, time
RUNNER_REV = "neumann-boundary-orbit-diagnostic-v2"
SOURCE_SHA = "108f30f954e3d807f16c4264fac56f4814c65ea4"
TRANSPORT_SHA = "a08a12b53bc0cfa4858e9eb04d06ea808694a9bc"
FILES = {
 "colab_neumann_boundary_orbit_diagnostic.py":"6b26d11c8d28080e5c993545b05a84c5feeb9f0bf2b62aee6bd4f57341c1b858",
 "verify_neumann_boundary_orbit_diagnostic.py":"6ebfaeffc62b1b416007dc02fc101ff2e541feff58355523bbae80322809283a",
 "verify_neumann_counting_reflection_diagnostic_v2.py":"a596e50eb708d39819da3f15557fa5705ab564874b70d9913f8a8b9794bec851"}
work = pathlib.Path("/content/orbit-diagnostic-v2-launch")
assert psutil.virtual_memory().total/2**30 >= 40, "HIGH_RAM_REQUIRED"
assert not pathlib.Path("/dev/nvidia0").exists(), "GPU_NOT_AUTHORIZED"
assert not work.exists(), "NO_REEXECUTION"
work.mkdir()
records = []
status = "FAIL"
inner = pathlib.Path("/content/hrpoly-"+RUNNER_REV+"-evidence.tar.gz")
outer = pathlib.Path("/content/orbit-diagnostic-v2-preservation-20260907.tar.gz")
def digest(p): return hashlib.sha256(p.read_bytes()).hexdigest()
def child(stage, cmd):
 start = time.perf_counter()
 log = work/(stage+".log")
 with log.open("xb") as stream:
  process = subprocess.Popen(cmd,stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
  print("STAGE="+stage+" PID="+str(process.pid),flush=True)
  code = process.wait()
 record = dict(stage=stage,command=cmd,exit=code,seconds=time.perf_counter()-start,log_sha256=digest(log))
 records.append(record)
 temp = work/"records.json.tmp"
 temp.write_text(json.dumps(records,sort_keys=True))
 temp.replace(work/"records.json")
 print(json.dumps(record),flush=True)
 print(log.read_text(errors="replace")[-5000:],flush=True)
 if code: raise RuntimeError("FIRST_ERROR="+stage)
try:
 for name,h in FILES.items():
  url="https://raw.githubusercontent.com/lluiseriksson/THE-ERIKSSON-PROGRAMME/"+TRANSPORT_SHA+"/scripts/"+name
  b=urllib.request.urlopen(url,timeout=60).read()
  assert hashlib.sha256(b).hexdigest()==h,"TRANSPORT_HASH="+name
  (work/name).write_bytes(b)
 print("HASH_GATE=PASS SOURCE_SHA="+SOURCE_SHA+" START_UTC="+datetime.datetime.now(datetime.timezone.utc).isoformat(),flush=True)
 child("diagnostic",["python3",str(work/"colab_neumann_boundary_orbit_diagnostic.py")])
 child("independent_reader",["python3",str(work/"verify_neumann_boundary_orbit_diagnostic.py"),"--archive",str(inner),"--sha256",digest(inner)])
 status="PASS"
except Exception as error:
 print(repr(error),flush=True)
finally:
 (work/"final.json").write_text(json.dumps(dict(status=status,source=SOURCE_SHA,transport=TRANSPORT_SHA,cold_seal=False),sort_keys=True))
 with tarfile.open(outer,"w:gz") as t:
  t.add(work,arcname=work.name)
  if inner.exists(): t.add(inner,arcname=inner.name)
 print("LAUNCH_FINAL_STATUS="+status+" COLD_SEAL=0",flush=True)
 print("PRESERVATION_ARCHIVE="+str(outer)+" SHA256="+digest(outer),flush=True)
 print("RUNTIME_RETAINED_FOR_EVIDENCE=1",flush=True)
